# SSH-sst feedback : 

Computes sea surface height anomalies (SSHA; used as a proxy for subsurface temperature) regressed onto sea surface temperature anomalies (SSTA) both in the eastern equatorial Pacific (horizontal Niño3 average).

#### Reference observations:
TropFlux and AVISO 1993-2018 (main)

SSH: JPL-MEASURES 1993-2021, CSIRO-SSH 1993-2019, GODAS 1980-2023, ORAS5 1958-2022, SODA3.4.2 1979-2018
SST: ERSSTv5 1854-2023, HadISST 1870-2023, COBE2 1850-2023, ERA5 1940-2022, 20CRv3 1836-2015, NCEP2 1979-2023


In [ ]:
from esmvalcore.dataset import Dataset

from esmvalcore.config import CFG

CFG["projects"]["OBS6"]["data"]["kj13"] = {
    "type": "esmvalcore.io.local.LocalDataSource",
    "rootpath": "/g/data/kj13/datasets/esmvaltool/obsdata-v2",
    "dirname_template": "Tier{tier}/{dataset}",
    "filename_template": "{project}_{dataset}_{type}_{version}_{mip}_{short_name}[_.]*nc",
}


In [ ]:
model_ssh_datasets = { 
"ACCESS-ESM1-6": 
    Dataset(
    short_name='zos',
    project="CMIP7",
    mip="ocean",
    exp="historical",
    ensemble="r1i1p1f1",
    timerange="19790101/20190101",
    dataset="ACCESS-ESM1-6",
    grid="g118",
    activity="CMIP",
    institute="ACCESS-Consortium",
    region="glb",
    frequency="mon",
    branding_suffix="tavg-u-hxy-sea"
)}

#tos
model_tos_datasets = {"ACCESS-ESM1-6": 
    model_ssh_datasets["ACCESS-ESM1-6"].copy(short_name='tos')}



obs_tos_datasets = {
"HadISST": 
    Dataset(
    short_name='tos',
    dataset='HadISST',
    mip="Omon",
    project='OBS',
    type='reanaly',
    version='1',
    timerange="19930101/20190101",
    tier=2),
# "TROPFLUX":
#     Dataset(
#     short_name='tos',
#     dataset='TROPFLUX',
#     mip="Omon",
#     project='OBS6',
#     type='reanaly',
#     tier=2),
"ERSSTv5":
    Dataset(
    short_name='tos',
    dataset='NOAA-ERSSTv5',
    mip="Omon",
    project='OBS6',
    type='reanaly',
    version='v5',
    tier=2),
}
obs_zos_datasets = {
 "CMEMS": 
     Dataset(
     short_name='zos',
     dataset='CMEMS',
     mip="Omon",
     project='OBS6',
     type='sat',
     version='SEALEVEL-GLO-PHY-L4',
     timerange="19930101/20190101",
     tier=2)
}

### Add Ancillary files

In [ ]:
for name, dataset in model_ssh_datasets.items():
    dataset.add_supplementary(short_name='sftof', mip="ocean", frequency="fx", branding_suffix="ti-u-hxy-u", grid="g118")
    dataset.add_supplementary(short_name="areacello", mip="ocean", frequency="fx", branding_suffix="ti-u-hxy-u", grid="g118")
for name, dataset in model_tos_datasets.items():
    dataset.add_supplementary(short_name='sftof', mip="ocean", frequency="fx", branding_suffix="ti-u-hxy-u", grid="g118")
    dataset.add_supplementary(short_name="areacello", mip="ocean", frequency="fx", branding_suffix="ti-u-hxy-u", grid="g118")

In [ ]:
model_ssh_datasets = {name: dataset.load() for name, dataset in model_ssh_datasets.items()}
model_tos_datasets = {name: dataset.load() for name, dataset in model_tos_datasets.items()}

obs_tos_datasets = {name: dataset.load() for name, dataset in obs_tos_datasets.items()}
obs_zos_datasets = {name: dataset.load() for name, dataset in obs_zos_datasets.items()}

In [ ]:
from esmvalcore.preprocessor import anomalies
from esmvalcore.preprocessor import area_statistics
from esmvalcore.preprocessor import seasonal_statistics
from esmvalcore.preprocessor import rolling_window_statistics
from esmvalcore.preprocessor import convert_units
from esmvalcore.preprocessor import extract_region
from esmvalcore.preprocessor import extract_season
from esmvalcore.preprocessor import extract_month
from esmvalcore.preprocessor import detrend
from esmvalcore.preprocessor import regrid
from esmvalcore.preprocessor import detrend
from esmvalcore.preprocessor import mask_landsea
from esmvalcore.preprocessor import extract_time
import iris

import matplotlib.pyplot as plt
import iris.quickplot as qplt
import numpy as np


In [ ]:
def ssh_east(cube):
    nino3_region = {"start_longitude": 210., "end_longitude": 270., "start_latitude": -5., "end_latitude": 5.}
    # cube = extract_time(cube, start_year=1979, end_year=2000, start_month=1, end_month=12, start_day=1, end_day=31)
    cube = mask_landsea(cube, mask_out="land")
    cube = convert_units(cube, units="cm")
    cube = extract_region(cube, **nino3_region)
    cube = anomalies(cube, period="monthly")

    cube = detrend(cube, dimension="time", method="linear")
    cube = regrid(cube, target_grid="1x1", scheme="linear")
    cube = area_statistics(cube,operator='mean')
    return cube


def tos_east(cube):
    nino3_region = {"start_longitude": 210., "end_longitude": 270., "start_latitude": -5., "end_latitude": 5.}

    cube = convert_units(cube, units="degC")
    cube = mask_landsea(cube, mask_out="land")
    cube = anomalies(cube, period="monthly")
    cube = detrend(cube, dimension="time", method="linear")
    cube = regrid(cube, target_grid="1x1", scheme="linear")    
    cube = extract_region(cube, **nino3_region)
    cube = area_statistics(cube,operator='mean')
    return cube   

In [ ]:
model_ssh_prep = {name: ssh_east(dataset) for name, dataset in model_ssh_datasets.items()}
model_tos_prep = {name: tos_east(dataset) for name, dataset in model_tos_datasets.items()}

obs_tos_prep = {name: tos_east(dataset) for name, dataset in obs_tos_datasets.items()}
obs_zos_prep = {name: ssh_east(dataset) for name, dataset in obs_zos_datasets.items()}

## Calculate Linear regression

In [ ]:

def linreg_1d(a_cube, b_cube):
    print("cube shapes:", a_cube.shape, b_cube.shape)
    B_data = b_cube.data
    B_with_intercept = np.vstack([B_data, np.ones_like(B_data)]).T
    coefs, _, _, _ = np.linalg.lstsq(B_with_intercept, a_cube.data, rcond=None)
    slope, intercept = coefs[0], coefs[1]
    return slope, intercept


# Diagnostic Level 1

In [ ]:

xseq = np.linspace(-40, 40, num=100)

slope, intcpt = linreg_1d(model_tos_prep["ACCESS-ESM1-6"], model_ssh_prep["ACCESS-ESM1-6"])
plt.plot(xseq, intcpt+slope*xseq)
print(slope, intcpt)
mod_slope = slope

slope, intcpt = linreg_1d(obs_tos_prep["HadISST"], obs_zos_prep["CMEMS"])
plt.plot(xseq, intcpt+slope*xseq, color='black')
metric_val = abs((mod_slope-slope)/slope)*100

print(model_tos_prep["ACCESS-ESM1-6"].shape, model_ssh_prep["ACCESS-ESM1-6"].shape) #scatter plot
print(model_tos_prep["ACCESS-ESM1-6"].units, model_ssh_prep["ACCESS-ESM1-6"].units)
plt.scatter(model_ssh_prep["ACCESS-ESM1-6"].data, model_tos_prep["ACCESS-ESM1-6"].data, s=10)

print(obs_tos_prep["HadISST"].shape, obs_zos_prep["CMEMS"].shape)
plt.scatter(obs_zos_prep["CMEMS"].data, obs_tos_prep["HadISST"].data, s=20, c='black', marker='D')
plt.xlim(-30,30)
yunit = model_tos_prep["ACCESS-ESM1-6"].units 
xunit = model_ssh_prep["ACCESS-ESM1-6"].units

plt.title('SSH to SST coupling')
plt.ylabel(f'nino3 SSTA ({yunit})')
plt.xlabel(f'nino3 SSHA ({xunit})')
plt.grid(linestyle='--')

## text slope
plt.text(0.02, 0.95, f'slope: {mod_slope:.2f} ', fontsize=10, ha='left', color='C0',
         transform=plt.gca().transAxes)
plt.text(0.02, 0.9, f'ref slope: {slope:.2f} ', fontsize=10, ha='left', color='k',
         transform=plt.gca().transAxes)

plt.text(0.99, 0.03, f'metric: {metric_val:.2f}%', fontsize=12, ha='right',
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
# print(metric_val)

# Diagnostic Level 2

In [ ]:
import pandas as pd

def linreg_df(var_a, var_b):
    B_data = np.array(var_b)
    B_with_intercept = np.vstack([B_data, np.ones_like(B_data)]).T
    coefs, _, _, _ = np.linalg.lstsq(B_with_intercept, np.array(var_a), rcond=None)
    slope, intercept = coefs[0], coefs[1]
    return slope, intercept

def plt_lvl2_subplot(y_cube, x_cube, dataset_label):
    
    df = pd.DataFrame({'var_a':y_cube.data, 'var_b':x_cube.data})
    slopes = []
    print(dataset_label, 'shape: ', df.shape)
    plt.scatter(x_cube.data, y_cube.data, c='k', s=10)
    xseq = np.linspace(-40, 40, num=50)
    slope, intcpt = linreg_df(df['var_a'], df['var_b'])
    plt.plot(xseq, intcpt+slope*xseq, c='black')
    slopes.append(slope)
    
    xseq = np.linspace(-40, 0, num=50) #
    slope, intcpt = linreg_df(df.loc[df['var_b']<0, 'var_a'], df.loc[df['var_b']<0, 'var_b'])
    
    plt.plot(xseq, intcpt+slope*xseq, linewidth=3)
    slopes.append(slope)
    
    xseq = np.linspace(0, 40, num=50)
    slope, intcpt = linreg_df(df.loc[df['var_b']>0, 'var_a'], df.loc[df['var_b']>0, 'var_b'])
    plt.plot(xseq, intcpt+slope*xseq, color='red', linewidth=3)
    slopes.append(slope)

    plt.title(dataset_label)
    plt_settings(slopes)

def plt_settings(slopes):
    plt.xlim(-40,40)
    plt.xticks(np.arange(-40,50,20))
    plt.ylim(-6,6)
    plt.yticks(np.arange(-6,7,3))
    plt.grid(linestyle='--')
    plt.xlabel(f'nino3 SSHA (cm)')
    plt.ylabel(f'nino3 SSTA (°C)')
    print(slopes)
    plt.text(0.02, 0.85, f'slope(all): {slopes[0]:.2f}\nslope(x<0): {slopes[1]:.2f}\nslope(x>0): {slopes[2]:.2f}', fontsize=12, ha='left',
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

In [ ]:
fig = plt.figure(figsize=(13, 5))
ax = plt.subplot(121)

plt_lvl2_subplot(model_tos_prep["ACCESS-ESM1-6"],model_ssh_prep["ACCESS-ESM1-6"], 'ACCESS-ESM1-6')

ax = plt.subplot(122)
plt_lvl2_subplot(obs_tos_prep["HadISST"], obs_zos_prep["CMEMS"], 'HadISST_CMEMS')

Figure 2: scatterplot of sea surface height anomalies (SSHA) and sea surface temperature anomalies (SSTA) in the eastern equatorial Pacific (Niño3 averaged), showing the possible nonlinearity in the strength of the SSH-to-SST coupling (usually shows no nonlinearity in both reference and model). The black, red and blue lines and numbers show respectively linear regression computed for all SSHA, SSHA>0 and SSHA<0, the left and right scatterplots show respectively the reference and the model.

## Dive down level 3

In [ ]:
#level 3 zonal runningaverage
from esmvalcore.preprocessor import meridional_statistics
from esmvalcore.preprocessor import mask_above_threshold, mask_below_threshold

def ssh_eqp(cube, name='model'):
    region = {"start_longitude": 150., "end_longitude": 270., "start_latitude": -5., "end_latitude": 5.}
    # cube = extract_time(cube, start_year=1979, end_year=2000, start_month=1, end_month=12, start_day=1, end_day=31)
    cube = mask_landsea(cube, mask_out="land")
    cube = convert_units(cube, units="cm")

    cube = extract_region(cube, **region)
    cube = anomalies(cube, period="monthly")
    cube = detrend(cube, dimension="time", method="linear")
    if name == 'obs':
        cube = regrid(cube, target_grid="1x1", scheme="linear") #issue on model, regrid needed on obs
    cube = area_statistics(cube, operator='mean')  
    
    return cube

    
def sst_meri_eqp(cube):
    region = {"start_longitude": 150., "end_longitude": 270., "start_latitude": -5., "end_latitude": 5.}

    cube = convert_units(cube, units="degC")
    cube = mask_landsea(cube, mask_out="land") 
    cube = anomalies(cube, period="monthly")
    cube = detrend(cube, dimension="time", method="linear")
    cube = regrid(cube, target_grid="1x1", scheme="linear")   
    cube = extract_region(cube, **region)
    cube = meridional_statistics(cube,operator='mean')
    cube = rolling_window_statistics(cube, coordinate='longitude',operator='mean',window_length=30)
    return cube 

In [ ]:
model_zos = {name: ssh_eqp(dataset) for name, dataset in model_ssh_datasets.items()}
model_tos = {name: sst_meri_eqp(dataset) for name, dataset in model_tos_datasets.items()}

obs_zos = {name: ssh_eqp(dataset, 'obs') for name, dataset in obs_zos_datasets.items()}
obs_tos = {name: sst_meri_eqp(dataset) for name, dataset in obs_tos_datasets.items()}

In [ ]:
def lin_regress_matrix(cubeA, cubeBsst):
    
    A_data = cubeA.data.reshape(cubeA.shape[0], -1)  # Shape (time, spatial_points)
    if cubeA.shape[0] == cubeBsst.shape[0]:
        B_data = cubeBsst.data.flatten() # or all
    else:
        B_data = cubeBsst.data.compressed() # masked threshold cube (time,) 

    # Add intercept term by stacking a column of ones with cubeB
    B_with_intercept = np.vstack([B_data, np.ones_like(B_data)]).T
    
    print('least squares data shapes ',B_with_intercept.shape, A_data.shape)
    # Solve the linear equations using least squares method
    coefs, _, _, _ = np.linalg.lstsq(B_with_intercept, A_data, rcond=None)

    # Create a new Iris Cube for the regression results
    result_cube = iris.cube.Cube(coefs[0], long_name='regression SSTA,SSHA',
                                 dim_coords_and_dims=[(cubeA.coord('longitude'), 0)])

    return result_cube


def feedback_nonlin(sst_cube, tauu_cube):
    tauu_aux = tauu_cube.copy()
    sst_coord = iris.coords.AuxCoord(sst_cube.data, sst_cube.standard_name, sst_cube.long_name, sst_cube.var_name, sst_cube.units)
    tauu_aux.add_aux_coord(sst_coord, 0)
    print(sst_cube.standard_name, sst_cube.long_name, sst_cube.var_name, sst_cube.units)

    below0 = iris.Constraint(sea_surface_height_above_geoid=lambda cell: cell < 0) #change variable
    above0 = iris.Constraint(sea_surface_height_above_geoid=lambda cell: cell > 0)
    ssta_neg = mask_above_threshold(sst_cube.copy(), 0) #x<0
    ssta_pos = mask_below_threshold(sst_cube.copy(), 0) #x>0
    xbelow0 = tauu_aux.extract(below0)
    xabove0 = tauu_aux.extract(above0)
    
    outreg_cube = lin_regress_matrix(xbelow0, ssta_neg)
    posreg_cube = lin_regress_matrix(xabove0, ssta_pos)

    return outreg_cube, posreg_cube

def format_longitude(x, pos):
    if x > 180:
        return f'{int(360 - x)}°W'
    elif x == 180:
        return f'{int(x)}°'
    else:
        return f'{int(x)}°E'
        

In [ ]:
## equator check numbers for zos...
print(f'min: {model_zos["ACCESS-ESM1-6"].data.min()}, {model_ssh_prep["ACCESS-ESM1-6"].data.min()} max: {model_zos["ACCESS-ESM1-6"].data.max()}, {model_ssh_prep["ACCESS-ESM1-6"].data.max()}')
obs_zos["CMEMS"].data.min(), obs_zos["CMEMS"].data.max()


In [ ]:
neg, pos = feedback_nonlin(model_zos["ACCESS-ESM1-6"], model_tos["ACCESS-ESM1-6"])

qplt.plot(neg, color='blue', linestyle='solid', label='SSHA<0')

qplt.plot(pos, color='red', linestyle='solid', label='SSHA>0')

neg, pos = feedback_nonlin(obs_zos["CMEMS"], obs_tos["HadISST"])
qplt.plot(neg, color='blue', linestyle='--')
qplt.plot(pos, color='red', linestyle='--')

cb = lin_regress_matrix(model_tos["ACCESS-ESM1-6"], model_zos["ACCESS-ESM1-6"])
        
qplt.plot(cb, color='black', linestyle='solid', label='model')

cb2 = lin_regress_matrix(obs_tos["HadISST"], obs_zos["CMEMS"])
qplt.plot(cb2, color='black', linestyle='--', label='HadISST_CMEMS')

plt.xlim(170, 250)
plt.xlabel('longitude')
plt.ylabel('reg(SSHA, SSTA)')
plt.grid(linestyle='--')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(format_longitude))
plt.legend()
plt.title("Thermocline feedback")

Figure 3: spatial structure of sea surface temperature anomalies (SSTA) regressed onto sea surface height anomalies (SSHA) both in the equatorial Pacific (meridional 5°S-5°N average; zonal 30° running average), showing the possible nonlinearity in the strength of the SSH-to-SST coupling (usually shows too strong coupling in the central and eastern equatorial Pacific, no nonlinearity in both reference and model; here the model shows a nonlinearity in the far eastern equatorial Pacific). The black, red and blue lines and numbers show respectively linear regression computed for all SSHA, SSHA>0 and SSHA<0, the dashed and solid curves show respectively the reference and the model.

## Dive down level 4:

In [ ]:
from esmvalcore.preprocessor import climate_statistics, extract_month
import iris.plot as iplt

def feedback_nonlin_lvl4(sst_cube, tauu_cube):
    tauu_aux = tauu_cube.copy()
    sst_coord = iris.coords.AuxCoord(sst_cube.data, sst_cube.standard_name, sst_cube.long_name, sst_cube.var_name, sst_cube.units)
    tauu_aux.add_aux_coord(sst_coord, 0)
    # print(sst_cube.standard_name, sst_cube.long_name, sst_cube.var_name, sst_cube.units)

    below0 = iris.Constraint(coord_values={sst_cube.standard_name:lambda cell: cell < 0})
    above0 = iris.Constraint(coord_values={sst_cube.standard_name:lambda cell: cell > 0})
    ssta_neg = mask_above_threshold(sst_cube.copy(), 0) #x<0
    ssta_pos = mask_below_threshold(sst_cube.copy(), 0) #x>0
    xbelow0 = tauu_aux.extract(below0)
    xabove0 = tauu_aux.extract(above0)
    
    outreg_cube = annual_structure_reg(xbelow0, ssta_neg)
    posreg_cube = annual_structure_reg(xabove0, ssta_pos)
    all_cube = annual_structure_reg(tauu_cube, sst_cube)

    return all_cube, posreg_cube, outreg_cube

def lin_regress_4(cubeA, cubeBsst):
    A_data = cubeA.data.reshape(cubeA.shape[0], -1)  # Shape (time, spatial_points)
    if cubeA.shape[0] == cubeBsst.shape[0]:
        B_data = cubeBsst.data.flatten() # or all
    else:
        B_data = cubeBsst.data.compressed() # compress masked

    B_with_intercept = np.vstack([B_data, np.ones_like(B_data)]).T

    coefs, _, _, _ = np.linalg.lstsq(B_with_intercept, A_data, rcond=None)
    return coefs[0]
    
def annual_structure_reg(nhf_cube, ts_cube):
    months_reg = []
    for i in range(1,13):
        #extract for both cubes
        nhf = extract_month(nhf_cube, month=i)
        ts = extract_month(ts_cube, month=i)
        coefs = lin_regress_4(nhf, ts)
        # collect array for months
        months_reg.append(coefs)

    # get month_number coordinate from an existing cube
    m_cube = climate_statistics(ts_cube, operator='max', period='month')
    
    #create cube, x = months, y = lon
    result_cube = iris.cube.Cube(np.array(months_reg), long_name='regression',
                                 dim_coords_and_dims=[(nhf.coord('longitude'), 1), (m_cube.coord('month_number'), 0)])
    return result_cube

def format_longitude(x, pos):
    if x > 180:
        return f'{int(360 - x)}°W'
    elif x == 180:
        return f'{int(x)}°'
    else:
        return f'{int(x)}°E'

In [ ]:
modells = feedback_nonlin_lvl4(model_zos["ACCESS-ESM1-6"], model_tos["ACCESS-ESM1-6"])
obsls = feedback_nonlin_lvl4(obs_zos["CMEMS"], obs_tos["HadISST"])
sub = ['', '>0', '<0',] #all_cube, posreg_cube, outreg_cube

fig = plt.figure(figsize=(8, 12))
i = 321
ax1 = plt.subplot(i)

for isub, cbls in enumerate(zip(obsls, modells)):
    # build ls 

    for cb in cbls:
        if i == 321:
            ax1.set_yticks(range(1, 13, 4))
            ax1.set_yticklabels(['Jan', 'May', 'Sep'] )
            ax1.set_ylabel('Months')
            ax1.set_title('ref: CMEMS_HadISST \nreg(SSHA, SSTA)', loc='left')
        else:
            ax2 = plt.subplot(i, sharex=ax1, sharey=ax1)
            ax2.set_title(f'reg(SSHA{sub[isub]}, SSTA)', loc='left')
            if i == 322:
                ax2.set_title('ACCESS-ESM1-6 \nreg(SSHA, SSTA)', loc='left')


        ## contour plt data
        cf1 = iplt.contourf(cb, coords=['longitude','month_number'], levels=np.arange(-0.3, 0.3, 0.02), extend='both', cmap="RdBu_r")
        
        if i < 325:
            plt.tick_params('x', labelbottom=False)
        else:
            plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(format_longitude)) #apply to bottom left =>325
            ax2.set_xlabel("longitude")
        
        if i%2 == 0:
            plt.tick_params(labelleft=False)
        
        i += 1


cax = plt.axes([0.15,0.01,0.7,0.03])
cbar = fig.colorbar(cf1, cax=cax, orientation='horizontal', extend='both', ticks=np.arange(-0.3, 0.31, 0.1))
cbar.set_label('regression (°C/cm)')

Figure 4: spatio-mean annual structure of sea surface temperature anomalies (SSTA) regressed onto sea surface height anomalies (SSHA) both in the equatorial Pacific (meridional 5°S-5°N average; zonal 30° running average), showing the possible nonlinearity in the strength of the SSH-to-SST coupling (usually shows too strong coupling in the central and eastern equatorial Pacific, no nonlinearity in both reference and model, too strong seasonality; here the model shows a nonlinearity in the far eastern equatorial Pacific). The first, second and third rows show respectively linear regression computed for all SSHA, SSHA>0 and SSHA<0, the left and right Hovmöllers show respectively the reference and the model.